# Measurement-Mechanistic RCA Usage

This notebook shows the high-level Python interface for four common workflows:

1. contaminated data without supplied priors;
2. contaminated data with supplied priors;
3. clean training data plus new samples;
4. causal discovery followed by RCA.

Run it from the repository root after `pip install -e .`.

In [ ]:
import networkx as nx
import numpy as np

from measurement_mechanistic_rca import MeasurementMechanisticRCA, discover_dag_nogam

In [ ]:
def make_chain_data(n=300, seed=0):
    rng = np.random.default_rng(seed)
    x0 = rng.normal(size=n)
    x1 = 0.8 * x0 + rng.normal(scale=0.2, size=n)
    x2 = np.sin(x1) + rng.normal(scale=0.2, size=n)
    return np.column_stack([x0, x1, x2])


true_dag = nx.DiGraph([(0, 1), (1, 2)])
clean_data = make_chain_data(n=300, seed=1)
contaminated_data = clean_data.copy()
contaminated_data[12, 1] += 4.0
contaminated_data[40, 0] += 3.0

measurement_priors = np.array([0.01, 0.10, 0.01])
generative_priors = np.array([0.05, 0.01, 0.01])

## 1. Contaminated Data, Priors Learned Implicitly

Use `fit_explain` directly when the input batch contains the anomalous samples and no external prior probabilities are available.

In [ ]:
model = MeasurementMechanisticRCA()
result = model.fit_explain(contaminated_data, true_dag)

result.top_features(sample_idx=12, k=3)

## 2. Contaminated Data With Supplied Priors

Priors can be global vectors with one probability per feature or sample-specific arrays with shape `(n_samples, n_features)`.

In [ ]:
model = MeasurementMechanisticRCA(
    prior_mode="single_root",
)
result_with_priors = model.fit_explain(
    contaminated_data,
    true_dag,
    measurement_priors=measurement_priors,
    generative_priors=generative_priors,
)

result_with_priors.top_features(sample_idx=12, k=3)

## 3. Clean Training Data, Then New Samples

Use `fit` on clean data to learn mechanisms and residual distributions. Then call `explain` on future samples. Priors are required in this setting because the clean training data does not contain anomaly frequencies.

In [ ]:
new_samples = make_chain_data(n=20, seed=2)
new_samples[3, 1] += 4.0

model = MeasurementMechanisticRCA()
model.fit(clean_data, true_dag)
future_result = model.explain(
    new_samples,
    measurement_priors=measurement_priors,
    generative_priors=generative_priors,
)

future_result.top_features(sample_idx=3, k=3)

## 4. Causal Discovery Followed by RCA

The RCA method expects a DAG. If the graph is not known, first learn one with a causal discovery method, then pass the learned `networkx.DiGraph` to `fit_explain` or `fit`/`explain`.

This example uses the bundled Python NoGAM implementation from the replication package. NoGAM has no R dependency; install the optional causal-discovery dependencies with `pip install -e ".[causal-discovery,notebooks]"`.

In [ ]:
learned_dag = discover_dag_nogam(
    clean_data,
    n_crossval=3,
    prune=True,
    alpha=0.05,
)
list(learned_dag.edges(data=True))

In [ ]:
model = MeasurementMechanisticRCA()
discovery_result = model.fit_explain(
    contaminated_data,
    learned_dag,
    measurement_priors=measurement_priors,
    generative_priors=generative_priors,
)

discovery_result.top_features(sample_idx=12, k=3)

## Backend Variants

The default `regression_backend="expectile"` is the Python spline expectile variant. Use `regression_backend="quantile"` for the scikit-learn spline median-regression replacement. Use `regression_backend="qgam"` only in an environment with R, qgam, mgcv, and rpy2 installed.